# BTS Digital Twin (NVS) — Private set1: train 1 scene / lần chạy

Notebook này chỉ xử lý **1 scene private duy nhất mỗi lần chạy** (biến `SCENE` ở
Bước 6). Có **8 scene private**: `HCM0249`, `HCM0254`, `HCM0276`, `HCM1439`, `HNI0131`, `HNI0265`, `HNI0366`, `HNI0437`.

**Chạy được trên cả Kaggle lẫn Google Colab** (tự nhận diện nền tảng ở Bước 1, biến
`WORKDIR`) — tiện luân phiên 2 nền tảng để đỡ tốn quota GPU của Kaggle.

Cách dùng: giống hệt notebook public — đổi `SCENE` ở Bước 6 rồi chạy lại (Kaggle: Save
Version; Colab: runtime mới), lặp lại cho 8 scene. Mỗi scene độc lập hoàn toàn (không
có "nền tảng" chung giữa các scene, cũng không liên quan gì tới các scene public).

**Khác với notebook public: KHÔNG có bước tính điểm PSNR/SSIM** — `private_set1` không
có ảnh ground-truth (BTC giữ lại để tự chấm), nên Bước 6 chỉ render rồi hiển thị vài
ảnh để tự mắt kiểm tra hợp lý (không bị nhiễu/méo/sai màu bất thường), không ra điểm số.

**Trước khi chạy, cần điền:**
1. Kaggle: Settings → Accelerator: **GPU T4 x2** (hoặc P100) → Internet: **On**.
   Colab: Runtime → Change runtime type → **GPU**.
2. `REPO_URL` ở Bước 3, `GDRIVE_URL` ở Bước 4 (đã điền sẵn), `SCENE` ở Bước 6.
3. 1 scene private mất khoảng **~2.5-3 giờ** (30000 iterations) — 1 phiên thừa sức
   xong 1 scene ở cả 2 nền tảng.

**Bảo mật:** để notebook này **Private**.


## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG có GPU — vào Settings bật Accelerator GPU trước khi chạy tiếp")
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)

Tự nhận diện nền tảng (Kaggle dùng `/kaggle/working`, Colab dùng `/content`) — mọi
cell sau đều dùng biến `WORKDIR` này thay vì viết cứng đường dẫn, để chạy được trên
cả 2 nền tảng mà không cần sửa gì (thuận tiện khi luân phiên Kaggle/Colab để đỡ tốn
quota GPU của Kaggle).


In [ ]:
import os

if os.path.isdir("/kaggle/working"):
    WORKDIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORKDIR = "/content"
else:
    WORKDIR = os.getcwd()

os.makedirs(WORKDIR, exist_ok=True)
print("Nền tảng phát hiện được — WORKDIR =", WORKDIR)


In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown

## Bước 2 — Clone + build 3D Gaussian Splatting

Repo gốc `graphdeco-inria/gaussian-splatting` — dùng để train/render, không tự viết lại
trainer (quá nhiều chi tiết dễ sai: densification, SH coefficients...). Bước build
2 CUDA extension (`diff-gaussian-rasterization`, `simple-knn`) mất khoảng 2-5 phút.

In [ ]:
%cd {WORKDIR}
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd {WORKDIR}/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
%cd {WORKDIR}
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = f"{WORKDIR}/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])


## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Nếu push CẢ project (gồm `Đề bài.md`, `KE_HOACH_VONG1.md`, `Dataset/`, `pipeline/`...)
làm 1 repo cũng được — cell dưới tự dò tìm thư mục con tên `pipeline` (chứa `common/`
và `scripts/`) ở bất kỳ độ sâu nào trong repo, không cần đúng ngay gốc repo.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin.git"
REPO_BRANCH = "feature/depth-anything-v2"  # <-- nhánh hướng đi #3 Depth Anything V2 (Hướng đi.md mục 2) trên nền Mip-Splatting; đổi thành "main" nếu chỉ cần baseline vanilla 3DGS

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

if not GITHUB_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
    except Exception:
        pass
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Colab Secrets.")
    except Exception:
        pass
if not GITHUB_TOKEN:
    print("Không tìm thấy secret 'GITHUB_TOKEN' ở Kaggle Secrets lẫn Colab Secrets "
          "(bỏ qua nếu repo Public, hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf {WORKDIR}/_repo_clone
!git clone --depth 1 --branch "{REPO_BRANCH}" "{clone_url}" {WORKDIR}/_repo_clone
print(f"Đã clone branch: {REPO_BRANCH}")

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về {WORKDIR}/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path(WORKDIR) / "_repo_clone"
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path(WORKDIR) / "pipeline"
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print(f"Đã symlink -> {target} ->", found.resolve())

Path(WORKDIR, "pipeline", "work").mkdir(parents=True, exist_ok=True)

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA/phase1/{public_set,private_set1}/...`
(zip nguyên thư mục `Dataset` như trong repo, hoặc chỉ riêng `VAI_NVS_DATA` cũng
được — cell dưới tự dò tìm thư mục `phase1` ở bất kỳ độ sâu nào trong zip).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/1GUflBBz4hrVtkMcHLfLMwUYk4HsJkXFu/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs(f"{WORKDIR}/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O {WORKDIR}/dataset.zip
!unzip -q -o {WORKDIR}/dataset.zip -d {WORKDIR}/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục phase1 ...")

In [ ]:
# Tự dò thư mục "phase1" (chứa public_set/ hoặc private_set1/) ở bất kỳ đâu trong
# zip vừa giải nén, rồi symlink về đúng vị trí mà pipeline/common/scenes.py cần:
#   {WORKDIR}/Dataset/VAI_NVS_DATA/phase1
import os
from pathlib import Path

RAW_ROOT = Path(WORKDIR) / "_dataset_raw"
found = None
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (public_set/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/phase1" trước "phase1" thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    p = Path(dirpath)
    if p.name == "phase1" and (("public_set" in dirnames) or ("private_set1" in dirnames)):
        found = p
        break

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'phase1' chứa public_set/private_set1 trong zip vừa giải nén.\n"
        f"Nội dung giải nén nằm ở {RAW_ROOT} — kiểm tra lại cấu trúc zip bạn đã upload lên Google Drive."
    )

print("Tìm thấy:", found)
target_parent = Path(WORKDIR) / "Dataset" / "VAI_NVS_DATA"
target_parent.mkdir(parents=True, exist_ok=True)
target = target_parent / "phase1"
if target.exists() or target.is_symlink():
    target.unlink() if target.is_symlink() else None
if not target.exists():
    os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

# QUAN TRỌNG: code (git clone) và dataset (Google Drive) không nằm chung 1 thư mục
# gốc trên Kaggle như lúc chạy local, nên common/scenes.py KHÔNG thể tự suy ra
# đường dẫn dataset bằng "đi lên N cấp từ vị trí file code" — phải khai báo thẳng
# qua biến môi trường này (đọc bởi pipeline/common/scenes.py).
os.environ["BTS_DATASET_ROOT"] = str(target.resolve())
print("Đã set BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 13 scene + scene nào có sparse hợp lệ — dataset đầy đủ
# thì kỳ vọng has_valid_provided_sparse=True cho CẢ 13 scene.
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại giá trị BTS_DATASET_ROOT ở cell
# trên có trỏ đúng chỗ chứa public_set/private_set1 hay không (thường do dataset.zip
# tải/giải nén thiếu — thử xoá {WORKDIR}/_dataset_raw và tải lại từ đầu).
import sys
sys.path.insert(0, f"{WORKDIR}/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.split:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")

## Bước 5 (tuỳ chọn) — Sanity-check hệ toạ độ

Script tự chạy lại COLMAP riêng cho `HCM0249` để so với sparse chính thức, xem có
khớp không — không bắt buộc, chỉ 1 lần đối chiếu cho chắc. Bật/tắt bằng `RUN_SANITY_CHECK`.

In [ ]:
RUN_SANITY_CHECK = False  # <-- đổi thành True khi muốn chạy lại bước đối chiếu này

if RUN_SANITY_CHECK:
    !python {WORKDIR}/pipeline/scripts/02_validate_frame.py
else:
    print("Bỏ qua sanity-check (RUN_SANITY_CHECK = False).")

## Bước 6 — Train + render 1 scene private

**Đổi `SCENE` thành 1 trong 8 tên sau rồi Save Version** (mỗi lần 1 tên, ở version khác nhau):

- `HCM0249`
- `HCM0254`
- `HCM0276`
- `HCM1439`
- `HNI0131`
- `HNI0265`
- `HNI0366`
- `HNI0437`

Chi tiết đầy đủ ghi ra file `pipeline/work/<scene>/03_train_3dgs.log`. Xem tiến độ lúc
train đang chạy: mở 1 cell khác gõ `!tail -n 30 <WORKDIR>/pipeline/work/<scene>/03_train_3dgs.log`
(thay `<WORKDIR>` bằng giá trị in ra ở Bước 1: `/kaggle/working` hoặc `/content`).

In [ ]:
SCENE = "HCM0249"  # <-- đổi thành tên scene private muốn train ở version này
!python {WORKDIR}/pipeline/scripts/01_run_colmap.py --scene {SCENE}

### Bước 6b — Sinh depth prior cho depth regularization (mặc định BẬT trên nhánh này)

Giống notebook public — mặc định BẬT (`USE_DEPTH_PRIOR = True`) trên nhánh
`feature/depth-anything-v2`. ⚠️ Trước khi dùng checkpoint train ở đây làm bản nộp thật
(`kaggle_submission.ipynb`), nhớ đã xác nhận depth prior cải thiện PSNR/SSIM/LPIPS rõ
ràng trên `public_set` trước — đúng quy trình "test public trước, roll-out private sau"
ở `Kết quả/Hướng đi.md` mục 5. Đổi `False` nếu muốn train baseline không depth prior để
so sánh cho scene này.


In [ ]:
USE_DEPTH_PRIOR = True  # mặc định BẬT trên nhánh feature/depth-anything-v2 — đổi False nếu muốn tắt tạm để so sánh

if USE_DEPTH_PRIOR:
    import os
    if not os.path.exists(f"{WORKDIR}/Depth-Anything-V2"):
        %cd {WORKDIR}
        !git clone https://github.com/DepthAnything/Depth-Anything-V2.git
        %cd {WORKDIR}/Depth-Anything-V2
        # Pin commit để tái lập được (đề bài mục 10.3) — commit mới nhất trên main đã kiểm tra
        # KHÔNG có .gitmodules (không submodule) và requirements.txt không ép version torch/
        # torchvision (an toàn, không đụng bản đã build cho gaussian-splatting ở Bước 2).
        !git checkout a561b849ebae10a6f5ef49e26c83cbbcd36c71bf
        %cd {WORKDIR}
        !pip install -q -r Depth-Anything-V2/requirements.txt
        !mkdir -p Depth-Anything-V2/checkpoints
        !wget -q -O Depth-Anything-V2/checkpoints/depth_anything_v2_vitl.pth \
            https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth
    os.environ["DA_REPO"] = f"{WORKDIR}/Depth-Anything-V2"
    !python {WORKDIR}/pipeline/scripts/08_generate_depth_priors.py --scene {SCENE}
else:
    print("Bỏ qua depth prior (USE_DEPTH_PRIOR=False) — DEPTH_PRIOR sẽ tự tắt ở cell train.")


In [ ]:
import os
os.environ["ITERATIONS"] = "30000"
os.environ["ANTIALIASING"] = "1"  # Mip-Splatting antialiasing — mặc định BẬT (Hướng đi.md mục 2 #2)
os.environ["DEPTH_PRIOR"] = "1" if USE_DEPTH_PRIOR else "0"  # khớp toggle ở cell Bước 6b phía trên
os.environ["EXPOSURE_COMP"] = "0"  # đổi "1" để bật --train_test_exp (Hướng đi.md mục 2 #5), mặc định tắt
!bash {WORKDIR}/pipeline/scripts/03_train_3dgs.sh {SCENE}


In [ ]:
!python {WORKDIR}/pipeline/scripts/04_render_test_poses.py --scene {SCENE}

**Không chạy `05_eval_metrics.py` ở đây** — private không có ảnh ground-truth để so
sánh (`has_valid_provided_sparse`/`gt_test_images_dir` chỉ có ở public_set, xem
`pipeline/common/scenes.py`). Cell dưới hiển thị vài ảnh render ra để tự mắt kiểm tra
hợp lý (không nhiễu loạn, không sai màu/hình dạng bất thường) trước khi lưu lên Drive.

In [ ]:
# Xem thử vài ảnh render ra (kiểm tra bằng mắt — không có điểm số vì private không có ảnh thật để so).
from pathlib import Path
from IPython.display import display
from PIL import Image

renders_dir = Path(f"{WORKDIR}/pipeline/work/{SCENE}/renders")
all_renders = sorted(renders_dir.glob("*.png"))
sample = all_renders[:4]
print(f"{len(all_renders)} ảnh render tại {renders_dir}, xem thử {len(sample)} ảnh đầu:")
for p in sample:
    display(Image.open(p))

## Bước 7 — Lấy checkpoint để lưu lên Google Drive

Không cần nén gì — tải thẳng cả thư mục rồi upload nguyên vậy lên Drive.

Checkpoint (trọng số đã train) nằm ở:
`pipeline/work/<SCENE>/gs_model/point_cloud/iteration_30000/point_cloud.ply`
(kèm 2 checkpoint giữa chừng ở `iteration_7000/` và `iteration_15000/`, phòng khi cần
iteration cuối bị lỗi).

Cách lấy (chỉ cần lấy đúng thư mục `pipeline/work/<SCENE>/gs_model/` — không cần lấy
nguyên `WORKDIR`, phần còn lại chỉ là code/dataset/repo clone, không cần cho submission):

- **Kaggle**: bấm **Save Version**, vào tab **Output**, tìm đúng thư mục đó, tải về máy.
- **Colab**: mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`)
  rồi copy thẳng thư mục đó vào Drive (`!cp -r {WORKDIR}/pipeline/work/{SCENE}/gs_model "/content/drive/MyDrive/..."`),
  hoặc tải trực tiếp về máy qua panel Files bên trái.

Sau đó upload thẳng lên Google Drive dưới dạng 1 thư mục — đặt tên thư mục trên Drive rõ
theo tên scene (vd `<SCENE>_gs_model`) để không nhầm lẫn khi điền link ở
`kaggle_submission.ipynb`. Nhớ đổi chế độ share thư mục đó thành "Anyone with the link".

Chạy xong 8 lần (8 version, 8 scene khác nhau) là đủ toàn bộ private_set1.
